In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

In [ ]:
#Making everything uniform

df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

df.head()

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
#For categorical features, replace them with 'NA'
#For numerical features, replace with with 0.0

categorical_features = df.select_dtypes(include=['object']).columns
numerical_features = df.select_dtypes(include=['number']).columns

df[categorical_features] = df[categorical_features].fillna('NA')
df[numerical_features] = df[numerical_features].fillna(0.0)

In [ ]:
df.isna().sum()

**Question 1**
**What is the most frequent observation (mode) for the column industry?**

In [ ]:
df['industry'].mode()[0]

**Question 2**
**Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation** **coefficient between every pair of features.**
**What are the two features that have the biggest correlation?**

In [ ]:
df[numerical_features].corr()

In [ ]:
# Split de data
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
#df_full_train: 80 % de los datos.
#df_test: 20 % de los datos.

df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

#df_train: 75 % de ese 80 % → 60 % del dataset original.
#df_val: 25 % de ese 80 % → 20 % del dataset original.

len(df_full_train), len(df_train), len(df_val), len(df_test)

In [ ]:
#Reseting the index of the dataframes

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
df_train.head()

**Question 3**  
**Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only**  
**Which of these variables has the biggest mutual information score?**  
- industry  
- location  
- lead_source  
- employment_status

In [ ]:
from sklearn.metrics import mutual_info_score

for col in categorical_features:
    score = mutual_info_score(df_train[col], df_train['converted'])
    print(col, round(score, 2))

In [ ]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

#Make sure that the target value converted is not in your dataframe.

del df_train['converted']
del df_val['converted']
del df_test['converted']

**Question 4**  
**Now let's train a logistic regression.**  
**Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.**  
**Fit the model on the training dataset.**  
**To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:**  
`model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`  
**Calculate the accuracy on the validation dataset and round it to 2 decimal digits.**  
**What accuracy did you get?**

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

categorical = df_train.select_dtypes(include=['object']).columns.tolist()
numerical = df_train.select_dtypes(include=['number']).columns.tolist()

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)

accuracy = (y_val == converted_decision).mean()
round(accuracy, 2)

**Question 5**  
**Let's find the least useful feature using the feature elimination technique.**  
**Train a model using the same features and parameters as in Q4 (without rounding).**  
**Now exclude each feature from this set and train a model without it. Record the accuracy for each model.**  
**For each feature, calculate the difference between the original accuracy and the accuracy without the feature.**  
**Which of following feature has the smallest difference?**  
- **'lead_source'**  
- **'number_of_courses_viewed'**  
- **'interaction_count'**  
**Note: The difference doesn't have to be positive.**

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

features = categorical + numerical

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
X_val = dv.transform(df_val[features].to_dict(orient='records'))

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
original_accuracy = (y_val == model.predict(X_val)).mean()

for feature in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    features_without = [f for f in features if f != feature]
    dv = DictVectorizer(sparse=False)
    X_train_without = dv.fit_transform(df_train[features_without].to_dict(orient='records'))
    X_val_without = dv.transform(df_val[features_without].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train_without, y_train)
    accuracy_without = (y_val == model.predict(X_val_without)).mean()

    print(feature, 'accuracy:', accuracy_without, 'difference:', original_accuracy - accuracy_without)

**Question 6**
**Now let's train a regularized logistic regression.**
**Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].**
**Train models using all the features as in Q4.**
**Calculate the accuracy on the validation dataset and round it to 3 decimal digits.**
**Which of these C leads to the best accuracy on the validation set?**

In [ ]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    accuracy = (y_val == model.predict(X_val)).mean()
    print(format(C, 'f'), round(accuracy, 3))